# NLP Analysis of AI Signals in 10-K Filings with Gemini

This portfolio project uses natural language processing and a large language model to extract company-level AI risks, opportunities, investment disclosures, and sentiment from ten Form 10-K filings.

The work originated in graduate coursework for **FNCE90084 Applications of Machine Learning in Finance at the University of Melbourne**. This public version has been restructured as a standalone portfolio notebook. Course instructions, grading material, and teaching templates have been removed. The analysis has been reorganised and revised for public presentation.

## Scope and workflow

The analysis covers NVIDIA, Alphabet, Microsoft, Amazon, Meta, Tesla, IBM, Intel, Salesforce, and Oracle. The pipeline:

1. reads each filing from HTML;
2. converts the filing to plain text;
3. prompts Gemini 2.0 Flash for structured AI-related information;
4. parses the response into risk, opportunity, investment, and investment-value fields;
5. consolidates the outputs in a pandas DataFrame; and
6. produces a high-level Bullish, Bearish, or Neutral AI sentiment label.

## Data source and reproducibility limits

The original analysis used ten company 10-K HTML files supplied inside the University of Melbourne course environment. Those files are **not included or embedded** in this public notebook. To reproduce the workflow, obtain the corresponding public filings from the SEC EDGAR database and place them in a local `data/` directory using the filename pattern `<TICKER>_latest_10K.html`.

The results summarised below come from the original analysis. They are not audited financial data, and generative-model responses may vary between runs.

## 1. Environment and model configuration

In [ ]:
import os
import re

import google.generativeai as genai
import pandas as pd
import tqdm
from bs4 import BeautifulSoup

GOOGLE_GEMINI_API_KEY = os.environ["GOOGLE_GEMINI_API_KEY"]
genai.configure(api_key=GOOGLE_GEMINI_API_KEY)
model = genai.GenerativeModel("gemini-2.0-flash")

The API key is loaded from an environment variable and is never stored in the notebook. The original coursework ran in a managed Jupyter environment with a secure key helper; the public version uses a standard local environment variable.

## 2. Load and clean a filing

In [ ]:
def load_html_file(file_path):
    """Load the complete contents of an HTML file as a string."""
    with open(file_path, "r", encoding="utf-8") as file:
        content = file.read()
    return content


def remove_html_tags(html_content):
    """Convert raw filing HTML to normalised plain text."""
    soup = BeautifulSoup(html_content, "html.parser")
    plain_text = soup.get_text(" ")
    plain_text = re.sub(r"http\S+", "", plain_text)
    plain_text = re.sub(r"\s+", " ", plain_text).strip()
    return plain_text


html_content = load_html_file("data/AMZN_latest_10K.html")
plain_text = remove_html_tags(html_content)

The raw filing and cleaned full text are intentionally not displayed. Avoiding embedded filing content keeps the public notebook compact and prevents redistribution of the course-provided dataset.

## 3. Structured extraction prompt

In [ ]:
def build_extraction_prompt(filing_text):
    return f"""
You are a financial analyst specialising in analysing company 10-K reports for AI-related risks, opportunities, and investments.

Read the company's 10-K report and summarise how the company views:
- AI-related Risks
- AI-related Opportunities
- AI-related Investments

For the investment section:
- Summarise any AI-related planned or committed investment mentioned by the company.
- Exclude any past or already executed investments. Do not add past and planned investments together.
- If a specific planned amount is given, express it clearly in billions of USD and include the latest year.
- Do not estimate information not supported by the report.

Use this structure:
**AI-related Risk:**
<2-3 concise sentences summarising AI risks>

**AI-related Opportunity:**
<2-3 concise sentences summarising AI opportunities>

**AI-related Investment:**
<2-3 concise sentences summarising investment information>
Investment Value (Billion USD): <number or null>
Investment Year (Latest year): <YYYY or null>

Your response must be factual and directly supported by the filing.

Here is the company's 10-K report text:
{filing_text}
"""


prompt = build_extraction_prompt(plain_text)

## 4. Generate and parse one company response

In [ ]:
response = model.generate_content(contents=prompt)
response_text = response.text
print(response_text)

In [ ]:
def parse_sections(response_text):
    risk = response_text.split("**AI-related Risk:**")[-1].split("**AI-related Opportunity:**")[0].strip()
    opportunity = response_text.split("**AI-related Opportunity:**")[-1].split("**AI-related Investment:**")[0].strip()
    investment = response_text.split("**AI-related Investment:**")[-1].strip()

    val_match = re.search(
        r"Investment Value \(Billion USD\):\s*([0-9]*\.?[0-9]+|null)",
        response_text,
        re.I,
    )
    if val_match:
        val_str = val_match.group(1).lower()
        investment_value = None if "null" in val_str else float(val_str)
    else:
        investment_value = None

    return pd.Series({
        "Risk": risk,
        "Opportunity": opportunity,
        "Investment": investment,
        "Investment Value (Billion USD)": investment_value,
    })


extracted_info = parse_sections(response_text)
display(extracted_info)

## 5. Scale the extraction across ten companies

In [ ]:
tickers = ["NVDA", "GOOGL", "MSFT", "AMZN", "META", "TSLA", "IBM", "INTC", "CRM", "ORCL"]
all_info = []

for tic in tqdm.tqdm(tickers, desc="Extracting information"):
    file_path = f"data/{tic}_latest_10K.html"
    html_content = load_html_file(file_path)
    plain_text = remove_html_tags(html_content)
    full_prompt = build_extraction_prompt(plain_text)
    response = model.generate_content(full_prompt)
    response_text = response.text
    all_info.append(parse_sections(response_text))

In [ ]:
df = pd.DataFrame(all_info, index=tickers)
df.index.name = "Ticker"
df.to_csv("10k_extracted_info.csv")
display(df)

### Empirical note

The original run produced structured company-level summaries and sentiment labels for all ten filings, demonstrating that the same extraction and parsing logic could be applied across a small document corpus. The numerical results are intentionally not foregrounded in this portfolio version: the main contribution is the NLP pipeline and the methodological lessons around context length, grounding, schema design, and reproducibility.

### Interpretation

The model broadly captured recurring themes in the filings, including competitive and regulatory risks, product and cloud opportunities, and infrastructure investment. The investment column is much less reliable as a cross-company measure. Many filings discuss general research, infrastructure, or capital expenditure without isolating an AI-only amount, while model outputs can vary across runs. A missing value therefore means that no explicit comparable amount was extracted; it does not mean zero AI investment.

## 6. Portfolio-level AI sentiment

In [ ]:
def format_company_info(tic, row):
    investment_value = row.get("Investment Value (Billion USD)")
    value_text = "null" if pd.isna(investment_value) else f"{investment_value}"
    return (
        f"{tic}:\n"
        f"- Risk: {row.get('Risk', '')}\n"
        f"- Opportunity: {row.get('Opportunity', '')}\n"
        f"- Investment: {row.get('Investment', '')}\n"
        f"- Investment Value (Billion USD): {value_text}\n"
    )


company_info = "\n".join(format_company_info(tic, df.loc[tic]) for tic in df.index)

ranking_prompt = f"""
Based only on the information provided, classify each company as Bullish, Bearish, or Neutral on AI.
Consider the balance of risks, opportunities, and investment. Do not use outside knowledge.
Return one company per line as: Ticker: Sentiment - brief rationale.

Company information:
{company_info}
"""

ranking_text = model.generate_content(ranking_prompt).text
print(ranking_text)

## 7. Limitations and next steps

- **Context length:** full filings are long, so chunking or retrieval would be more robust than sending an entire filing in one prompt.
- **Evidence grounding:** the extraction should require a filing section, chunk ID, and supporting quotation for every claim.
- **Validation:** quoted evidence should be checked programmatically against the source text before a claim is accepted.
- **Structured output:** a JSON schema would be safer than delimiter-based parsing.
- **Comparability:** capital expenditure, R&D, commitments, and strategic investments should be represented as separate fields.
- **Reproducibility:** model version, temperature, prompt version, and run timestamp should be logged.

A stronger production pipeline would first segment filings by SEC item, retrieve only relevant passages, request evidence-backed JSON, and reject any quotation that cannot be found in the source.

## Responsible-use note

This notebook is an educational portfolio artifact, not investment advice. Generative-model outputs must be verified against the original SEC filing before being used in research or decision-making. The underlying course dataset and teaching materials are not distributed here.